### Setup

In [1]:
# Install 'uv' for lightning-fast dependency resolution
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = f"{os.environ['HOME']}/.local/bin:{os.environ['PATH']}"

# Install PyTorch and ML libraries using uv
!uv pip install --system torch==2.6.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!uv pip install --system transformers==4.57.6 optimum==1.26.0 datasets==4.4.0 bitsandbytes peft wandb huggingface_hub

# Clone and install LLaMA-Factory
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory
!git checkout 762b480131908d37736ad9aa3f12e87f8f7e6313
!uv pip install --system -e .
!uv pip install --system -r requirements/metrics.txt

downloading uv 0.11.1 x86_64-unknown-linux-gnu
installing to /root/.local/bin
  uv
  uvx
everything's installed!

To add $HOME/.local/bin to your PATH, either restart your shell or run:

    source $HOME/.local/bin/env (sh, bash, zsh)
    source $HOME/.local/bin/env.fish (fish)
WARN: The following commands are shadowed by other commands in your PATH: uv uvx
Using Python 3.12.6 environment at: /usr/local
Resolved 28 packages in 551ms
⠙ Preparing packages... (0/18)
⠙ Preparing packages... (0/18)
⠙ Preparing packages... (0/18)
⠙ Preparing packages... (0/18)
⠙ Preparing packages... (0/18)
⠙ Preparing packages... (0/18)
sympy                  ------------------------------ 40.74 KiB/5.90 MiB
⠙ Preparing packages... (0/18)
sympy                  ------------------------------ 40.74 KiB/5.90 MiB
⠙ Preparing packages... (0/18)
sympy                  ------------------------------ 40.74 KiB/5.90 MiB
⠙ Preparing packages... (0/18)
sympy                  ------------------------------ 40.74 KiB/5

#### Authentication

In [ ]:
import wandb
from huggingface_hub import login

HF_TOKEN = "" 
WANDB_TOKEN = ""

login(token=HF_TOKEN)
wandb.login(key=WANDB_TOKEN)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: mo2004hashraf (mo2004hashraf-alexandria-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

#### Fetching and Preparing the Dataset

In [3]:
print("Installing Git LFS...")
!apt-get update -y -q && apt-get install git-lfs -y -q
!git lfs install

print("\nCleaning up old placeholder folder...")
!rm -rf ./dataset

print("Cloning public 3.4k dataset via Git LFS...")
# This time, it will download the actual gigabytes of data!
!git clone https://huggingface.co/datasets/mohamedashraff22/arabic-menus-dataset ./dataset

print("Dataset Downloaded Successfully!")

Installing Git LFS...
Get:1 http://deb.debian.org/debian bookworm InRelease [151 kB]
Get:2 http://deb.debian.org/debian bookworm-updates InRelease [55.4 kB]
Get:3 http://deb.debian.org/debian-security bookworm-security InRelease [48.0 kB]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/debian12/x86_64  InRelease [1581 B]
Get:5 http://deb.debian.org/debian bookworm/main amd64 Packages [8792 kB]
Get:6 http://deb.debian.org/debian-security bookworm-security/main amd64 Packages [294 kB]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/debian12/x86_64  Packages [1731 kB]
Fetched 11.1 MB in 2s (5036 kB/s)
Reading package lists...
Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  git-lfs
0 upgraded, 1 newly installed, 0 to remove and 118 not upgraded.
Need to get 3489 kB of archives.
After this operation, 11.2 MB of additional disk space will be used.
Get:1 http://deb.debian.org/debian

#### Registering Data with LLaMA-Factory

In [4]:
import json
import os

# The absolute path where the images just downloaded
true_image_dir = "/root/LLaMA-Factory/dataset/images"
train_file = "./dataset/train.json"
val_file = "./dataset/val.json"

print("Fixing image paths to local container...")

# 1. Update paths in both JSON files
for file_path in [train_file, val_file]:
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    for item in data:
        filename = os.path.basename(item["images"][0])
        # Overwrite the Google Drive path with the absolute container path
        item["images"][0] = os.path.join(true_image_dir, filename)
        
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

# 2. Register datasets inside LLaMA-Factory
dataset_info_path = "data/dataset_info.json"
with open(dataset_info_path, "r", encoding="utf-8") as f:
    ds_info = json.load(f)

# Add our new datasets
ds_info["ocr_train_3k"] = {
    "file_name": "../dataset/train.json",
    "formatting": "sharegpt",
    "columns": {"messages": "conversations", "images": "images"}
}
ds_info["ocr_val_3k"] = {
    "file_name": "../dataset/val.json",
    "formatting": "sharegpt",
    "columns": {"messages": "conversations", "images": "images"}
}

with open(dataset_info_path, "w", encoding="utf-8") as f:
    json.dump(ds_info, f, ensure_ascii=False, indent=2)
    
print("Paths fixed and datasets registered perfectly!")

Fixing image paths to local container...
Paths fixed and datasets registered perfectly!


#### YAML Configuration

In [5]:
yaml_content = """
### model
model_name_or_path: Qwen/Qwen2.5-VL-3B-Instruct
trust_remote_code: true

### method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 16
lora_target: all
lora_dropout: 0.1

### dataset
dataset: ocr_train_3k
eval_dataset: ocr_val_3k
template: qwen2_vl
cutoff_len: 4096
overwrite_cache: true
preprocessing_num_workers: 8

### output
output_dir: ./checkpoint_output/
logging_steps: 10
save_steps: 100
save_total_limit: 8
plot_loss: true
overwrite_output_dir: true

### train
per_device_train_batch_size: 1
gradient_accumulation_steps: 8
learning_rate: 2e-5
num_train_epochs: 2.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true
ddp_timeout: 180000000
optim: adamw_torch

### eval
per_device_eval_batch_size: 1
eval_strategy: steps
eval_steps: 100

report_to: wandb
run_name: modal-arabic-ocr-3k-run
"""

os.makedirs("examples/train_lora", exist_ok=True)
with open("examples/train_lora/modal_train.yaml", "w") as f:
    f.write(yaml_content.strip())
    
print("YAML Configuration generated. Launching training...")

# Ensure we are in the correct directory
os.chdir("/root/LLaMA-Factory")


YAML Configuration generated. Launching training...


#### Cleaning

In [6]:
import json
import os

train_file = "./dataset/train.json"
val_file = "./dataset/val.json"

print("Hunting for ghost images in the JSONs...\n")

for file_path in [train_file, val_file]:
    if not os.path.exists(file_path):
        continue
        
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        
    cleaned_data = []
    missing_count = 0
    
    for item in data:
        img_path = item["images"][0]
        # Only keep the entry if the image physically exists on the disk
        if os.path.exists(img_path):
            cleaned_data.append(item)
        else:
            missing_count += 1
            print(f"👻 Removed missing image from JSON: {os.path.basename(img_path)}")
            
    # Save the cleaned, crash-proof JSON
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(cleaned_data, f, ensure_ascii=False, indent=2)
        
    print(f"\n✅ {file_path} Cleaned! Kept: {len(cleaned_data)} | Removed: {missing_count}\n")

Hunting for ghost images in the JSONs...

👻 Removed missing image from JSON: menu_63_31.JPG
👻 Removed missing image from JSON: menu_59_7.JPG
👻 Removed missing image from JSON: menu_19_11.JPG
👻 Removed missing image from JSON: menu_79_12.JPG
👻 Removed missing image from JSON: menu_53_4.JPG
👻 Removed missing image from JSON: menu_59_18.JPG
👻 Removed missing image from JSON: menu_53_13.JPG
👻 Removed missing image from JSON: menu_53_16.JPG
👻 Removed missing image from JSON: menu_9_19.JPG
👻 Removed missing image from JSON: menu_59_31.JPG

✅ ./dataset/train.json Cleaned! Kept: 2762 | Removed: 10

👻 Removed missing image from JSON: menu_53_21.JPG

✅ ./dataset/val.json Cleaned! Kept: 307 | Removed: 1



#### Launch Training

In [7]:
# Launch!
!export DISABLE_VERSION_CHECK=1 && llamafactory-cli train examples/train_lora/modal_train.yaml

[WARNING|2026-03-25 21:05:18] llamafactory.extras.misc:155 >> Version checking has been disabled, may lead to unexpected behaviors.
/usr/local/lib/python3.12/site-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.12/site-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.12/site-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")
/usr/local/lib/python3.12/site-packages/rouge_chinese/rouge.py:93: SyntaxWarning: invalid escape sequence '\?'
  para = re.sub('([。！？\?])([^”’])', r"\1\n\2", para)
/usr/local/lib/python3.12/site-packages/rouge_chinese/rouge.py:94: SyntaxWarning: invalid escape sequence '\.'
  para = re.sub('(\.{6})([^”’])', r"\1\n\2", para)
/usr/local/lib/python3.12/site-packa

#### Push Checkpoints to New Repository

In [8]:
from huggingface_hub import HfApi

api = HfApi()
REPO_ID = "mohamedashraff22/arabic-menu-ocr-v2"

# Creates the repo if it does not exist
api.create_repo(repo_id=REPO_ID, private=False, exist_ok=True)

print(f"Uploading intermediate checkpoints to {REPO_ID}/checkpoints ...")
api.upload_folder(
    folder_path="./checkpoint_output/",
    repo_id=REPO_ID,
    repo_type="model",
    path_in_repo="checkpoints"
)
print("Checkpoints secured on Hugging Face!")

Uploading intermediate checkpoints to mohamedashraff22/arabic-menu-ocr-v2/checkpoints ...


Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  ...utput/checkpoint-100/tokenizer.json:  97%|#########7| 11.1MB / 11.4MB            

  ..._output/checkpoint-200/optimizer.pt:   0%|          |  129kB /  240MB            

...

  ..._output/checkpoint-100/scheduler.pt: 100%|##########| 1.06kB / 1.06kB            

  ..._output/checkpoint-300/optimizer.pt:   0%|          |  129kB /  240MB            

  ..._output/checkpoint-200/scheduler.pt: 100%|##########| 1.06kB / 1.06kB            

  ...nt_output/adapter_model.safetensors:   9%|8         | 10.7MB /  120MB            

Checkpoints secured on Hugging Face!


#### Merge LoRA Weights

In [9]:
import os

# Automatically find the latest checkpoint folder
checkpoints = [d for d in os.listdir("./checkpoint_output/") if d.startswith("checkpoint-")]
latest_checkpoint = sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
adapter_path = f"./checkpoint_output/{latest_checkpoint}"

print(f"Preparing to merge using adapter: {adapter_path}")

merge_yaml = f"""
### model
model_name_or_path: Qwen/Qwen2.5-VL-3B-Instruct
adapter_name_or_path: {adapter_path}
template: qwen2_vl
trust_remote_code: true

### export
export_dir: ./merged_model/
export_size: 5
export_device: cpu
export_legacy_format: false
"""

# Save the YAML configuration
os.makedirs("examples/merge_lora", exist_ok=True)
with open("examples/merge_lora/merge.yaml", "w") as f:
    f.write(merge_yaml.strip())

print(f"Merging Base Model with {latest_checkpoint}... (This requires high RAM)")

# Execute the merge command
!export DISABLE_VERSION_CHECK=1 && llamafactory-cli export examples/merge_lora/merge.yaml

Preparing to merge using adapter: ./checkpoint_output/checkpoint-692
Merging Base Model with checkpoint-692... (This requires high RAM)
[WARNING|2026-03-25 22:15:38] llamafactory.extras.misc:155 >> Version checking has been disabled, may lead to unexpected behaviors.
[INFO|tokenization_utils_base.py:2111] 2026-03-25 22:15:44,458 >> loading file vocab.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-VL-3B-Instruct/snapshots/66285546d2b821cf421d4f5eb2576359d3770cd3/vocab.json
[INFO|tokenization_utils_base.py:2111] 2026-03-25 22:15:44,458 >> loading file merges.txt from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-VL-3B-Instruct/snapshots/66285546d2b821cf421d4f5eb2576359d3770cd3/merges.txt
[INFO|tokenization_utils_base.py:2111] 2026-03-25 22:15:44,458 >> loading file tokenizer.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-VL-3B-Instruct/snapshots/66285546d2b821cf421d4f5eb2576359d3770cd3/tokenizer.json
[INFO|tokenization_utils_bas

#### Push Final Merged Model

In [11]:
from huggingface_hub import HfApi

api = HfApi()
REPO_ID = "mohamedashraff22/arabic-menu-ocr-v2"

print(f"Uploading final merged model weights to {REPO_ID}...")
api.upload_folder(
    folder_path="./merged_model/",
    repo_id=REPO_ID,
    repo_type="model"
)

print(f"🎉 Pipeline complete! Your fully fine-tuned model is live at: https://huggingface.co/{REPO_ID}")

Uploading final merged model weights to mohamedashraff22/arabic-menu-ocr-v2...


Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  ...Factory/merged_model/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

  ...el/model-00002-of-00002.safetensors:   1%|          | 15.6MB / 2.51GB            

  ...el/model-00001-of-00002.safetensors:   1%|          | 40.8MB / 5.00GB            

🎉 Pipeline complete! Your fully fine-tuned model is live at: https://huggingface.co/mohamedashraff22/arabic-menu-ocr-v2
